
# NCBI / PubMed / PMC + E-utilities Tutorial 

## What is NCBI, PubMed, and PMC?

* **NCBI** = *National Center for Biotechnology Information*

  * A U.S. government organization (part of the **NIH**) that **hosts many biomedical databases** and tools.
  * NCBI is the **platform/organization** — it is not a single database.
  * NCBI:

    * builds & maintains databases
    * hosts web interfaces (human-facing pages)
    * provides APIs (E-utilities)
    * maintains links between databases (e.g., PubMed ↔ PMC)

* **PubMed** (NCBI database)

  * A database of **biomedical citations and abstracts**.
  * Primary identifier: **PMID** (PubMed ID)

* **PMC (PubMed Central)** (NCBI database)

  * A database of **full-text articles** (when available in PMC).
  * Primary identifier: **PMCID** (e.g., `PMC7274568`)

### How PubMed and PMC are related

They are **different databases**, but they are **linked**.

* A paper can be:

  * ✅ in **PubMed only** (citation/abstract exists, no PMC full text)
  * ✅ in **PubMed + PMC** (citation + full text in PMC)
* A paper **cannot be in PMC without being in PubMed** (PMC articles are also indexed in PubMed)

---

## What is an API?

An **API (Application Programming Interface)** is a set of rules that allows software to communicate and exchange data.

### Case 1: URL that serves a web page (human-facing)

Example:

* `https://www.ncbi.nlm.nih.gov/pubmed/32720671`

What happens:

* The server generates an **HTML page** meant for humans (layout, styling, buttons, links).
* Python would still receive it as **text**, but it is not designed for programmatic extraction.

### Case 2: URL that is an API endpoint (machine-facing)

Example:

* `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi`

What happens:

* The server reads **query parameters** (inputs)
* Queries internal databases
* Returns **structured data** (XML, and sometimes JSON for some endpoints)

---

## What “using an API in a browser” really means

When you paste an API URL into a browser, you are:

* sending a **GET request** with parameters
* reading the raw response

You cannot interact with it (no buttons). The only control you have is editing the URL.

---

## URL parameters: what `?` and `&` mean

* `?` starts the query parameters section

  * everything after `?` is “inputs” to the server
* `&` separates parameters

Example:

```
https://example.com/path?key1=value1&key2=value2&key3=value3
```

---

## NCBI E-utilities (EUtils): the main endpoints

NCBI provides a family of APIs called **E-utilities**. Each endpoint does a different job:

| Endpoint        | Purpose                          |
| --------------- | -------------------------------- |
| `esearch.fcgi`  | Search (get IDs like PMIDs)      |
| `efetch.fcgi`   | Download full records (XML/text) |
| `esummary.fcgi` | Summary metadata                 |
| `elink.fcgi`    | Link records across databases    |

### Key idea: `elink.fcgi`

`elink.fcgi` = “Tell me how one database record connects to another database”

For example:

* **PMID → PMCID** (PubMed → PMC)

---

## Why APIs beat HTML scraping (for lots of PMIDs)

HTML scraping might work for a single quick lookup, but APIs are:

* faster
* stable (less likely to break when website layout changes)
* designed for automation

---

# Tutorial: Checking if a PMID has a PMCID (ELink)

## Step 1: Build the ELink request

We want:

* source database = PubMed
* target database = PMC
* only the *PubMed → PMC* link type (not “cites”, “related”, etc.)

### Parameters (important)

| Parameter                       | Meaning                            |
| ------------------------------- | ---------------------------------- |
| `dbfrom=pubmed`                 | start from PubMed records          |
| `db=pmc`                        | find linked PMC records            |
| `id=<PMID>`                     | the PubMed ID(s) you query         |
| `linkname=pubmed_pmc`           | restrict to PubMed→PMC links       |
| `retmode=xml` or `retmode=json` | output format (endpoint-dependent) |

---

## Step 2A: ELink returning **JSON** (then `r.json()` works)

```python
import requests

pmid = "11053615"

params = {
    "dbfrom": "pubmed",
    "db": "pmc",
    "id": pmid,
    "linkname": "pubmed_pmc",
    "retmode": "json",
}

r = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi",
    params=params,
)

data = r.json()
linksets = data.get("linksets", [])

has_pmc = False
pmc_ids = []

if linksets:
    linksetdbs = linksets[0].get("linksetdbs", [])
    if linksetdbs:
        pmc_ids = linksetdbs[0].get("links", [])
        if pmc_ids:
            has_pmc = True

print("has_pmc:", has_pmc)
print("pmc_ids:", pmc_ids)
```

### What this means

* `has_pmc=True` → the PMID has at least one linked PMCID
* `pmc_ids` → list of numeric PMC IDs (you can turn into `PMCxxxxx` if needed)

---

## Step 2B: ELink returning **XML** (then parse with ElementTree)

```python
import requests
import xml.etree.ElementTree as ET

pmid = "11053615"

params = {
    "dbfrom": "pubmed",
    "db": "pmc",
    "id": pmid,
    "linkname": "pubmed_pmc",
    "retmode": "xml",
}

r = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi",
    params=params,
)

root = ET.fromstring(r.text)

# Explore all Id tags (first 20)
ids = [el.text for el in root.findall(".//Id")]
print(ids[:20])
```

### Note

In XML output, `<Id>` can appear in multiple places:

* the input PMID
* linked IDs (PMC IDs)

So **context matters** (which parent tag it belongs to).

---

## Step 3: What you do next (pipeline idea)

Typical workflow:

1. Start from a PMID list
2. Use **ELink** to map PMID → PMCID
3. If PMCID exists, use **EFetch** with `db=pmc` to fetch **full text XML**
4. Extract text from `<body> → <sec> → <p>`

---

## Quick reminders

* `efetch.fcgi` is primarily **XML/text**, not JSON.
* `elink.fcgi` can provide **XML** and (in many cases) **JSON**.
* PubMed IDs are **PMID**; PMC full-text IDs are **PMCID**.
